# Fine-Tune Whisper on the Bisaya Speech Corpus (Kaggle Notebook)

This is a Kaggle-notebook adaptation of Hugging Face's
[Fine-Tune Whisper for Multilingual ASR](https://huggingface.co/blog/fine-tune-whisper)
tutorial, pointed at this project's own Bisaya (Cebuano) speech corpus
instead of the tutorial's original Hindi/Common Voice example. It trains
a third ASR system for this project's benchmark, alongside the Kaldi
HMM-GMM model (`train_kaldi.ipynb`) and ElevenLabs Scribe
(`evaluate_elevenlabs.ipynb`) -- see the repo's `README.md` for the rest
of that pipeline. This notebook is standalone: it doesn't feed into
`compare.ipynb` automatically.

**Differences from the original Colab tutorial**, besides the dataset:
- GPU/internet setup, Hugging Face auth, and long-run persistence all use
  Kaggle's own mechanisms instead of Colab's (Section "Kaggle Setup" below).
- The corpus has to be **uploaded to Kaggle as a Dataset first** (Section
  "Upload the Corpus to Kaggle" below) -- Kaggle notebooks can't read your
  local filesystem the way Colab can read from Google Drive or a direct
  download.
- The train/test split reuses the exact same speaker-independent split
  (by speaker, ~80/20, fixed seed 42) that `train_kaldi.ipynb` uses, so
  Whisper's held-out test speakers match Kaldi's -- a prerequisite for any
  later three-way comparison.
- Whisper has no dedicated Cebuano/Bisaya language token (~99 languages
  are supported; Bisaya isn't one). This notebook uses `"Tagalog"`
  (Whisper's closest Philippine-language code, `tl`) for the
  tokenizer/generation-config language conditioning -- a real
  approximation, not an exact match, called out again where it's used.


## Kaggle Setup

1. **Enable a GPU**: notebook Settings (right sidebar) -> Accelerator ->
   GPU T4 x2 (or P100/other, if available). Kaggle currently grants a
   weekly GPU-hour quota per account -- check your remaining quota in
   Settings before starting a long run.
2. **Enable internet access**: Settings -> Internet -> On. Required for
   `pip install`, downloading the pretrained Whisper checkpoint, and
   pushing to the Hugging Face Hub.
3. **Attach the corpus dataset**: see "Upload the Corpus to Kaggle" below
   -- do this once, then attach it via Add Input on every notebook that
   needs it.
4. **Add your Hugging Face token as a Kaggle Secret**: Add-ons -> Secrets
   -> add a secret named `HF_TOKEN` with a Hugging Face
   [write access token](https://huggingface.co/settings/tokens) as the
   value. Used in "Hugging Face Authentication" below instead of the
   interactive `notebook_login()` widget, so the notebook can run
   unattended (Save & Run All).

## Upload the Corpus to Kaggle

One-time step, done outside this notebook, before it can run:

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) -> **New
   Dataset**.
2. Upload every file under this project's `data/bisaya_audio/` (the
   Parquet shards) -- drag-and-drop in the browser, or use the
   [Kaggle API](https://www.kaggle.com/docs/api) from the machine that
   has the corpus locally:
   ```bash
   pip install kaggle
   # ~/.kaggle/kaggle.json holds your API credentials (Kaggle account ->
   # Settings -> Create New Token)
   kaggle datasets init -p data/bisaya_audio
   # edit the generated dataset-metadata.json: set a title/id, e.g.
   #   "id": "your-kaggle-username/bisaya-audio-corpus"
   kaggle datasets create -p data/bisaya_audio
   ```
3. In this notebook (or any Kaggle notebook that needs the corpus): **Add
   Input** (right sidebar) -> search for the dataset you just created ->
   Add. It appears under `/kaggle/input/<dataset-slug>/`.
4. Set `CORPUS_DIR` in the next cell to match wherever your Parquet files
   actually land under `/kaggle/input/`.

This corpus is not public -- keep the uploaded Kaggle Dataset **Private**
unless you have the right to publish it.

## Prepare Environment

In [1]:
!nvidia-smi


Mon Aug 24 05:25:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
pip install -q --force-reinstall "numpy<2.0.0" "scipy<1.14.0" "protobuf>=5.29.1,<6.0.0" pyarrow datasets transformers accelerator peft librosa soundfile subprocess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Python >=3.8,<3.11; 1.8.0rc3 Requires-Python >=3.8,<3.11; 1.8.0rc4 Requires-Python >=3.8,<3.11; 1.8.1 Requires-Python >=3.8,<3.11;

In [ ]:
import os

print("Environment setup complete! Restarting kernel...")
os._exit(00)

In [1]:
import os, subprocess

print("Installing system FFmpeg and setting audio decoding backend...")

# 1. Install system FFmpeg libraries required by C-extension audio decoders
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg",
    shell=True,
    check=True,
)

# 2. Disable torchcodec explicitly so datasets uses soundfile/librosa
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"

print("FFmpeg installed and torchcodec disabled!")

os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"

Installing system FFmpeg and setting audio decoding backend...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


FFmpeg installed and torchcodec disabled!


### Hugging Face Authentication

Uses the `HF_TOKEN` Kaggle Secret set up above, rather than the
interactive `notebook_login()` widget the original Colab tutorial uses --
this keeps the notebook runnable unattended via Kaggle's **Save & Run
All (Commit)**.

In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# Login and expose as environment variable for all Hugging Face libraries
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

## Load Dataset

Reads every Parquet shard in `CORPUS_DIR` via 🤗 `datasets` directly --
the corpus's `audio` column is already stored in `datasets`' own Audio
struct format (`{bytes, path}`, with the feature type recorded in the
Parquet file's own schema metadata), so `load_dataset("parquet", ...)`
decodes it automatically, the same way it would for a published HF
dataset like the tutorial's Common Voice.

In [3]:
from pathlib import Path

# Adjust to match the slug/path your attached Kaggle Dataset actually uses.
CORPUS_DIR = Path("/kaggle/input/datasets/troymerales/bisaya-audio")

parquet_files = sorted(str(p) for p in CORPUS_DIR.glob("*.parquet"))
assert parquet_files, f"No Parquet files found under {CORPUS_DIR} -- check the dataset is attached (Add Input)."
print(f"Found {len(parquet_files)} Parquet shard(s)")


Found 11 Parquet shard(s)


In [4]:
from datasets import load_dataset

raw_dataset = load_dataset("parquet", data_files=parquet_files, split="train")
print(raw_dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['audio', 'speaker_id', 'language', 'transcript', 'transcript_type', 'gender', 'country', 'mother_tongue', 'dialect', 'os', 'device', 'duration', 'script_type', 'words', 'n_words', 'transcript_model', 'aligner', 'age_band', 'native_speaker', 'proficiency'],
    num_rows: 90
})


### Speaker-Independent Train/Test Split

Same split `train_kaldi.ipynb` Section 8 uses -- by speaker (not
utterance), ~80/20, fixed seed 42 -- so this notebook's held-out test
speakers are the same ones Kaldi never trained on. Reused verbatim rather
than re-derived, so the split stays identical if this cell or
`train_kaldi.ipynb`'s changes independently.

In [5]:
from datasets import DatasetDict
import pandas as pd

SPLIT_SEED = 42


def speaker_independent_split(speakers, test_fraction=0.1, seed=SPLIT_SEED):
    speakers = sorted(speakers)
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])
    assert train_speakers.isdisjoint(test_speakers)
    return train_speakers, test_speakers


all_speakers = set(raw_dataset.unique("speaker_id"))
train_speakers, test_speakers = speaker_independent_split(all_speakers)

# Construct DatasetDict directly without using a placeholder split
bisaya = DatasetDict(
    {
        "train": raw_dataset.filter(
            lambda ex: ex["speaker_id"] in train_speakers
        ),
        "test": raw_dataset.filter(
            lambda ex: ex["speaker_id"] in test_speakers
        ),
    }
)

print(f"seed = {SPLIT_SEED}")
print(
    f"train: {len(train_speakers)} speakers, {len(bisaya['train'])} audio files"
)
print(f"test:  {len(test_speakers)} speakers, {len(bisaya['test'])} audio files")

Filter:   0%|          | 0/90 [00:00<?, ? examples/s]

Filter:   0%|          | 0/90 [00:00<?, ? examples/s]

seed = 42
train: 13 speakers, 77 audio files
test:  2 speakers, 13 audio files


In [6]:
# Keep audio and transcript columns
keep_cols = {"audio", "transcript"}
bisaya = bisaya.remove_columns(
    [c for c in bisaya["train"].column_names if c not in keep_cols]
)
print("Columns after Cell 14 pruning:", bisaya["train"].column_names)
# Output should be: ['audio', 'transcript']

Columns after Cell 14 pruning: ['audio', 'transcript']


## Prepare Feature Extractor, Tokenizer and Data

### Load WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

`language="Tagalog"` is Whisper's closest built-in Philippine-language
token (`tl`) -- there is no Cebuano/Bisaya entry in Whisper's ~99-language
list. This is an approximation the model was not specifically designed
for; treat any language-conditioning benefit from it as best-effort, not
exact.

In [7]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

MODEL_CHECKPOINT = "openai/whisper-small"
LANGUAGE = "Tagalog"  # closest Whisper-supported code to Bisaya/Cebuano -- see note above

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

### Prepare Data

In [8]:
print(bisaya["train"][0]["transcript"])

Niadto ako og picnic niadto miaging bulan sa usa ka matahom nga Sabado sa buntag. Gikauban ko sa akong pamilya ang pag-adto sa sapa sa among bukid. Sa kadapit nga dagan og kahoy ug bugnaw ang hangin. Gisugdan namo ang among adlaw sa sayong pagpamahaw. Si mama nag-andam og adobo ug sina ngag, nga among gidala sa mga plastic nga sudlanan. Ako ug ang akong manghud nag-amping sa paghakot sa banig, unlan ug usa ka basket nga puno og prutas. Dili mi kalimot sa termos nga puno og init nga tsokolate. Pag-abot namo sa sapa, nasurpresa mi sa katingalahang tan-awon. Ang tubig sa sapa hayag og limpyo ug ang mga langgam nagkanta sa mga sanga sa kahoy. Gipahiluna namo ang among banig sa ubos sa dako nga acacia tree diin adunay maanindot nga landong. Ang among adlaw napuno og kalipay. Naglupad-lupad mi og agipo, nagsakay sa balsa nga hinimo sa kawayan ug naligo sa bugnaw nga tubig sa sapa. Si papa nag-istorya bahin sa iyang kabataan sa samang dapit ug miuli kami nga adunay bag-ong pahinumdum nga amon

In [9]:
print(bisaya.column_names)

{'train': ['audio', 'transcript'], 'test': ['audio', 'transcript']}


In [10]:
from datasets import Audio

# 1. Ensure 16kHz audio sampling
bisaya = bisaya.cast_column("audio", Audio(sampling_rate=16000))


# 2. Extract Mel-spectrograms & truncate labels safely to 448
def prepare_dataset(batch):
    audio = batch["audio"]
    array = audio["array"]
    sr = audio["sampling_rate"]

    # Truncate audio array to 30 seconds max (480,000 samples at 16kHz)
    max_samples = 30 * sr
    if len(array) > max_samples:
        array = array[:max_samples]

    # Extract log-Mel features
    batch["input_features"] = feature_extractor(
        array, sampling_rate=sr
    ).input_features[0]

    # Truncate text tokens to Whisper's max limit of 448
    batch["labels"] = tokenizer(
        batch["transcript"], max_length=448, truncation=True
    ).input_ids

    return batch


# 3. Drop all current columns automatically (prevents KeyError/ValueError)
bisaya = bisaya.map(
    prepare_dataset,
    remove_columns=bisaya["train"].column_names,  # Dynamic drop list
    num_proc=1,  # Single-threaded to avoid Kaggle freezes
)

print("Processed dataset columns:", bisaya["train"].column_names)
# Expected output: ['input_features', 'labels']

Map (num_proc=1):   0%|          | 0/77 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/13 [00:00<?, ? examples/s]

Processed dataset columns: ['input_features', 'labels']


## Training and Evaluation

Same 🤗 Trainer-based pipeline as the original tutorial: load a
pretrained checkpoint, define a data collator, define the WER metric,
configure and run training.

### Load a Pre-Trained Checkpoint

In [11]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)
model.generation_config.language = LANGUAGE.lower()
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

### Define a Data Collator

In [12]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad pre-extracted log-Mel input_features
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label token sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Mask padding tokens in labels with -100
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Strip initial decoder start token if present
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# OVERWRITE the old data_collator in memory
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

### Evaluation Metrics

In [13]:
pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.0 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [26]:
import evaluate
import numpy as np

# Load Word Error Rate metric
metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # 1. Replace -100 in labels with pad_token_id so decoding doesn't crash or output empty strings
    label_ids = np.where(
        label_ids != -100, label_ids, tokenizer.pad_token_id
    )

    # 2. Decode token IDs back to human-readable text
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # 3. Compute WER metric
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

### Define the Training Configuration

`push_to_hub=True` is the main persistence strategy on Kaggle, same as
the original tutorial recommends for Colab: Kaggle notebook sessions are
also ephemeral (working-directory contents don't survive past the
session unless explicitly saved), so periodic checkpoints pushed to the
Hub protect the run against an interrupted or killed session.

In [15]:
print("Train dataset column names:", bisaya["train"].column_names)
print("First item keys:", bisaya["train"][0].keys())

Train dataset column names: ['input_features', 'labels']
First item keys: dict_keys(['input_features', 'labels'])


In [16]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-bisaya-lora",  # Repositories with LoRA are best tagged clearly
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch size = 16
    learning_rate=1e-3,  # Boosted to 1e-3 for LoRA adapters (use 1e-5 ONLY if full fine-tuning without LoRA)
    warmup_ratio=0.1,  # Uses 10% of total steps for warmup instead of a fixed 500 steps
    num_train_epochs=5,  # Replaces max_steps; loops through your 3.14h train set 5 times (~75 total steps)
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",  # Evaluates WER once at the end of every epoch (Replaces deprecated evaluation_strategy)
    save_strategy="epoch",  # Saves a checkpoint once at the end of every epoch
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=5,  # Reduced logging interval since total steps will be < 100
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
from transformers import Seq2SeqTrainer

# Re-instantiate Collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Re-instantiate Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=bisaya["train"],
    eval_dataset=bisaya["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

### Training

**Running unattended on Kaggle:** unlike Colab (which needs the browser
tab open and a JS keep-alive trick to survive idle disconnects), Kaggle
runs a notebook as a real background batch job when you click **Save
Version -> Save & Run All (Commit)** -- you can close the browser and it
keeps training. Use interactive "Edit" mode only for iterating on the
early cells; switch to a commit run for the actual training job.

Watch your GPU-hour quota -- a run that hits Kaggle's session time limit
partway through is still recoverable via `push_to_hub`'s periodic
checkpoints (re-load with `from_pretrained` on your Hub repo and resume),
but budgeting `max_steps` to fit inside one session avoids that entirely.

In [18]:
# Launch fine-tuning run
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Wer
1,No log,14.535919,100.000000
2,30.195630,13.348575,127.268643
3,30.195630,9.159286,100.000000
4,41.521417,8.074327,100.000000
5,24.294798,7.401149,100.000000


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=15, training_loss=32.00394846598307, metrics={'train_runtime': 282.9797, 'train_samples_per_second': 1.361, 'train_steps_per_second': 0.053, 'total_flos': 1.111053791232e+17, 'train_loss': 32.00394846598307, 'epoch': 5.0})

Update `kwargs` to match your run, then push the model card + final
checkpoint to the Hub. `dataset_tags`/`dataset_args` are omitted since
this corpus isn't a public Hub dataset (see "Upload the Corpus to
Kaggle" above) -- don't set them to a dataset id that doesn't exist.

In [20]:
kwargs = {
    "dataset": "Bisaya Speech Corpus (private)",
    "language": "ceb",  # ISO 639-3 for Cebuano/Bisaya -- Hub model-card metadata only;
                         # unrelated to Whisper's own "Tagalog" language token used above
    "model_name": "Whisper Small Bisaya",  # a 'pretty' name for our model
    "finetuned_from": MODEL_CHECKPOINT,
    "tasks": "automatic-speech-recognition",
}


In [21]:
trainer.push_to_hub(**kwargs)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/troxyz1268/whisper-small-bisaya-lora/commit/b32cc37088c7575a340061a1a52964f3ac4487f2', commit_message='End of training', commit_description='', oid='b32cc37088c7575a340061a1a52964f3ac4487f2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/troxyz1268/whisper-small-bisaya-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='troxyz1268/whisper-small-bisaya-lora'), pr_revision=None, pr_num=None)

# Export for local machine

In [23]:
# Save the model and processor explicitly from the trainer
save_directory = "./whisper_bisaya_final"

trainer.save_model(save_directory)
processor.save_pretrained(save_directory)

print("Saved model and processor to:", save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model and processor to: ./whisper_bisaya_final


In [ ]:
import shutil
from IPython.display import FileLink

# 1. Zip the output folder
shutil.make_archive("whisper_bisaya_model", "zip", save_directory)

# 2. Display the download link
print("Click below to download your fine-tuned Whisper model:")
FileLink(r"whisper_bisaya_model.zip")

In [27]:
eval_results = trainer.evaluate(eval_dataset=bisaya["test"])
print("Evaluation Results:", eval_results)

Evaluation Results: {'eval_loss': 14.535919189453125, 'eval_wer': 100.0, 'eval_runtime': 10.6203, 'eval_samples_per_second': 1.224, 'eval_steps_per_second': 0.094, 'epoch': 5.0}


In [28]:
import pandas as pd
import torch

# 1. Grab first 5 test samples
sample_batch = bisaya["test"].select(range(min(5, len(bisaya["test"]))))

# 2. Extract input features and convert to PyTorch tensors
input_features = [
    torch.tensor(ex["input_features"]) for ex in sample_batch
]
input_features = torch.stack(input_features).to(model.device)

# 3. Generate predictions using Whisper model
with torch.no_grad():
    generated_ids = model.generate(
        input_features,
        max_new_tokens=225,
        language="tl",  # Force language context
        task="transcribe",
    )

# 4. Decode predictions and true ground-truth labels
pred_texts = tokenizer.batch_decode(
    generated_ids, skip_special_tokens=True
)

label_ids = [ex["labels"] for ex in sample_batch]
# Replace -100 padding with pad_token_id for safe decoding
label_ids = [
    [t if t != -100 else tokenizer.pad_token_id for t in l]
    for l in label_ids
]
ref_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

# 5. Display side-by-side comparison table
comparison_df = pd.DataFrame(
    {
        "Reference (Ground Truth)": ref_texts,
        "Model Prediction": pred_texts,
    }
)

pd.set_option("display.max_colwidth", None)
display(comparison_df)

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,Reference (Ground Truth),Model Prediction
0,"Unsa imong paboritong nga app? Ang ako gyung paborito nga app no kay Twitter, Discord o kaning Facebook. Di man jud namawala ang Facebook pero usahay toxic lang ka-- toxic lang kaayo ang naa sa Facebook. Pero ganahan ko Discord kay naa man didto ang mga projects nga akong gipang-apilan kaning mga online sama sa ani nga airdrops, mga ingon ana. Ug diri sa Telegram kay naa man diri ako mga kauban sa nag kuan ani ubang project. So, samot nang Twitter, ganahan kaay ko sa Twitter kay dali ra siya ba. Unya daghan kaay didtong mga tweets nga akong ganahan. Niyá open kaayo siya sa tanan. Open siya sa mga bashers, mga unsa pa dinha naa syay ug fake news gani ang imohang makuanan, naay mga, naay mga community nga motudlo nimo nga fake news na siya ug i-i-korek ka sa komunidad. Ingon ana ba.",
1,"Unsa ang imong paborito nga bulak? Daghan kay ko'g paborito nga bulak, pero ang rose usa na na siya. Unya unsa gani toy? Chrysanthemum man siguro na pero gan-- ambot ganahan kay ko'g chrysanthemum nga yellow kay ako gibutang sa altar. Pero ang rose, nindot man gud kaayong rose kay basta nindot kaayo siya tan-awon sambot naglain-lain og kolor. Mga orchids ingon ana, nindot kaayo na sila. Ganahan kay ko ana nila. Unya ang mga orchids, mga dahlia, mga cosmos, basta daghan. nya karon ay nindot kaayo ning sakura pero ang sakura tua raman sa Japan wala man kaayo ni diri. Cherry blossoms mao raman siguro nag sakura? Ganahan kay ikog ana nila.",
2,"Kanus-a nimo labhan ang imong bayo? Ang pasabot siguro ani nga bayo kay murag deeper man ni siya nga Bisaya or Cebuano version. Dili man ni siya sa amoang nga pagka-Cebuana diri sa Cebu. So, kani siya, ang akong bayo or ang akong mga sinina, labhan naku siguro kaduha sa usa ka semana. Pero karon sa, nga busy kaayo ko, ako na ning ipa-laundry. Unya kay karon gidaghan man ni kaayo, mga pila siguro ni siya ka load sa laundry kon ako ni siya ipalabhan didto sa gawas. Unya kay magpalaba man ko karon, ang ako siguro mabayran ani mga sa tulo ka load naa na siya sa mga 600 kapin. Unya mopalit lang ko og Downy sa gawas, tag diyes ang usa, unya kinahanglan kon tulo ga ni kay maggamit man ko og Downy ana upat kada load. So, upat kada load, unya tulo ka, tulo ka, tulo ka hulganan, tulo ka load, tag upat kada, kada load, so naay 12 ang akong palitunon ani nga Downy. Unya ang sabon nga gamiton ani nga liquid soap kay Kuan, tag, tag duha kada lunod. Unya mopalit pagyud ko og kanang, unsa gitawag ani, bleach ba, kanang zonrox ani nga dekulor. Bahinon lang naku katong tag baynte. Bahinon sa tulo ka load. Unya inig kahuman sa tulo ka load, mga duha ka oras mahuman na ang labha, unya ako pa ni siyang ipapiko daan didto. Unya t",
3,"Unsa oras ka kasagaran mubangon? Kasagaran ko mubangon kana sigurong alas sais na. Alas sais na ko kasagaran mubangon. Pero sa una katong mag-jogging jogging pa ko katong mag-measure pa ko sa silencio mubangon ko mga alas singko kay mag-jogging pa man ko ana, mag-walking ko bisan asa kay mag-mag-mangita ko og kanang mga hex bitaw kay mag-measure ko sa mga hex, mag-measure kog mga noise. diay dili hex, mga noise. Niya, diri sa Selencio. Niya, karon nga wala na to siya, voice AI na lang, gitapol na sad ko og laag-laag. Tapol na ko og lakaw-lakaw so mag-record na lang ko diri pareha ani karon. Mag-record na lang ko. Sa sayo gihapon ko mumatag kay mag-record man ko sayo sa buntag pero dili na ko maglakaw-lakaw.",
4,"Pilan na ka katuig sa trabaho? Taud-taud na kong nagtrabaho ug dugay na kaayo ko nga nasigig trabaho og mga napulo na ka tuig kong naadtoan sa trabaho. Bisan unsa na lang nga klase sa trabaho ang akong gisudlan. Ngano maning mga ingon ani mani? Tungod kay lisod kaayo ang among kahimtang, so kinahanglan gyud nga magtrabaho aron makakuha og kwarta. Kay kon walay kwarta, unsay may among ipalit og bugas? Og usa imo'y palit og bugas, kon wala may bugas, unsa may among kaonon? So kinahanglan gyud nga manarbaho mi para na ka may sweldo. Unya wala ko kasa

In [29]:
import torch

# Test a single sample from test set
sample = bisaya["test"][0]
input_features = (
    torch.tensor(sample["input_features"]).unsqueeze(0).to(model.device)
)

with torch.no_grad():
    generated_ids = model.generate(
        input_features,
        max_new_tokens=50,
    )

print("Generated Token IDs:", generated_ids.cpu().numpy().tolist())
print(
    "Decoded raw output:",
    tokenizer.decode(generated_ids[0], skip_special_tokens=False),
)

Both `max_new_tokens` (=50) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Token IDs: [[50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265]]
Decoded raw output: <|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|>


In [30]:
import torch

# 1. Obtain official prompt token IDs for Tagalog/Bisaya transcription
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="tl", task="transcribe"
)

# 2. Extract a test sample
sample = bisaya["test"][0]
input_features = (
    torch.tensor(sample["input_features"]).unsqueeze(0).to(model.device)
)

# 3. Generate predictions with forced decoder prompt IDs
with torch.no_grad():
    generated_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids,
        max_new_tokens=225,
    )

# 4. Decode
raw_decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
clean_decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Raw Decoded Tokens:", raw_decoded)
print("Clean Prediction:", clean_decoded)

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw Decoded Tokens: <|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|fr|><|

## Closing Remarks

This notebook fine-tunes Whisper small on this project's Bisaya corpus
using 🤗 Datasets, Transformers, and the Hugging Face Hub, adapted for
Kaggle's environment (GPU/internet setup, Kaggle Secrets for
authentication, dataset upload, and unattended long-run training via
Save & Run All). See the original
[fine-tuning blog post](https://huggingface.co/blog/fine-tune-whisper)
for the underlying theory, and this project's own `README.md`/`CLAUDE.md`
for how this fits alongside the Kaldi and ElevenLabs systems in the wider
benchmark.